In [0]:
%sql
-- Reset Gold
TRUNCATE TABLE _exponent.omop_scm.observation_period;


In [0]:
%sql
-- Reset Silver
DELETE FROM _exponent.omop_silver.observation_period
WHERE source_system = 'allscripts_scm';


In [0]:
%sql
-- Reset Mapping
DELETE FROM _exponent.omop_mapping.source_to_observation_period
WHERE source_system = 'allscripts_scm';


### Source Tables:
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3order` — all order activity
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit` — visit/encounter dates
- `_exponent._bronze_allscripts_scm_prod_01.dbo_cv3client` — for DeceasedDtm capping

### Strategy:
- Aggregate MIN/MAX clinical activity dates per patient across orders and visits
- Create one observation period per patient (no gap splitting)
- Cap end date at DeceasedDtm if patient is deceased
- Use period_type_concept_id = 32817 (EHR encounter record)

### Notes:
- This notebook depends on source_to_person being populated for allscripts_scm
- Observation period represents the span of time during which the patient has data in the EHR
- Date filtering: >= 1900-01-01 and <= CURRENT_DATE to exclude invalid dates


In [0]:
%sql
-- Create silver_observation_period temp view for SCM
CREATE OR REPLACE TEMPORARY VIEW silver_observation_period AS

WITH clinical_activity AS (
  -- Orders (all types)
  SELECT
    ClientGUID,
    COALESCE(RequestedDtm, Entered, CreatedWhen) AS activity_date
  FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order
  WHERE Active = TRUE
    AND ClientGUID IS NOT NULL
    AND COALESCE(RequestedDtm, Entered, CreatedWhen) IS NOT NULL

  UNION ALL

  -- Client Visits
  SELECT
    ClientGUID,
    COALESCE(AdmitDtm, CreatedWhen) AS activity_date
  FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3clientvisit
  WHERE Active = TRUE
    AND ClientGUID IS NOT NULL
    AND COALESCE(AdmitDtm, CreatedWhen) IS NOT NULL
),

-- Aggregate per patient: MIN start, MAX end
patient_observation_window AS (
  SELECT
    ClientGUID,
    MIN(DATE(activity_date)) AS observation_start,
    MAX(DATE(activity_date)) AS observation_end
  FROM clinical_activity
  WHERE activity_date >= '1900-01-01'
    AND activity_date <= CURRENT_DATE()
  GROUP BY ClientGUID
)

-- Final select with death date capping
SELECT
  pow.observation_start AS observation_period_start_date,
  CASE
    WHEN c.DeceasedDtm IS NOT NULL AND DATE(c.DeceasedDtm) < pow.observation_end
    THEN DATE(c.DeceasedDtm)
    ELSE pow.observation_end
  END AS observation_period_end_date,
  32817 AS period_type_concept_id,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(pow.ClientGUID AS STRING)) AS person_source_value,
  CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(pow.ClientGUID AS STRING)) AS observation_period_source_value,
  'allscripts_scm' AS source_system
FROM patient_observation_window pow
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3client c
  ON pow.ClientGUID = c.GUID
INNER JOIN _exponent.omop_mapping.source_to_person stp
  ON stp.person_source_value = CONCAT_WS(CHR(31), 'allscripts_scm', 'cv3client', 'GUID', CAST(pow.ClientGUID AS STRING))
  AND stp.active_flag = TRUE

In [0]:
# %sql
# -- Preview
# SELECT * FROM silver_observation_period LIMIT 10

In [0]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.observation_period AS t
USING silver_observation_period AS s
ON t.observation_period_source_value = s.observation_period_source_value

WHEN MATCHED AND (
     NOT (t.observation_period_start_date <=> s.observation_period_start_date)
  OR NOT (t.observation_period_end_date <=> s.observation_period_end_date)
  OR NOT (t.period_type_concept_id <=> s.period_type_concept_id)
  OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.observation_period_start_date = s.observation_period_start_date,
  t.observation_period_end_date   = s.observation_period_end_date,
  t.period_type_concept_id        = s.period_type_concept_id,
  t.person_source_value           = s.person_source_value,
  t.source_system                 = s.source_system,
  t.last_mod_tsp                  = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  observation_period_start_date,
  observation_period_end_date,
  period_type_concept_id,
  person_source_value,
  observation_period_source_value,
  source_system,
  last_mod_tsp
)
VALUES (
  s.observation_period_start_date,
  s.observation_period_end_date,
  s.period_type_concept_id,
  s.person_source_value,
  s.observation_period_source_value,
  s.source_system,
  current_timestamp()
);

In [0]:
# %sql
# -- Verify silver
# SELECT * FROM _exponent.omop_silver.observation_period
# WHERE source_system = 'allscripts_scm'
# LIMIT 10

In [0]:
%sql
-- Insert new mappings to source_to_observation_period
INSERT INTO _exponent.omop_mapping.source_to_observation_period (
    source_system,
    observation_period_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.observation_period_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, observation_period_source_value, last_mod_tsp
    FROM _exponent.omop_silver.observation_period
    WHERE source_system = 'allscripts_scm'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_observation_period x
  ON s.observation_period_source_value = x.observation_period_source_value;

In [0]:
%sql
-- Merge to Gold layer
-- MERGE INTO _exponent.omop.observation_period AS gold
MERGE INTO _exponent.omop_scm.observation_period AS gold
USING (
  SELECT
    sop.observation_period_id,
    stp.person_id,
    s.observation_period_start_date,
    s.observation_period_end_date,
    s.period_type_concept_id
  FROM _exponent.omop_silver.observation_period s
  JOIN _exponent.omop_mapping.source_to_observation_period sop
    ON sop.observation_period_source_value = s.observation_period_source_value
   AND sop.active_flag = TRUE
  JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = s.person_source_value
   AND stp.active_flag = TRUE
  WHERE s.source_system = 'allscripts_scm'
) AS src
ON gold.observation_period_id = src.observation_period_id

WHEN MATCHED THEN UPDATE SET
  gold.person_id                       = src.person_id,
  gold.observation_period_start_date   = src.observation_period_start_date,
  gold.observation_period_end_date     = src.observation_period_end_date,
  gold.period_type_concept_id          = src.period_type_concept_id

WHEN NOT MATCHED THEN INSERT (
  observation_period_id,
  person_id,
  observation_period_start_date,
  observation_period_end_date,
  period_type_concept_id
)
VALUES (
  src.observation_period_id,
  src.person_id,
  src.observation_period_start_date,
  src.observation_period_end_date,
  src.period_type_concept_id
);

In [0]:
# %sql
# -- Layer counts
# SELECT 'Silver' AS layer, COUNT(*) AS record_count FROM _exponent.omop_silver.observation_period WHERE source_system = 'allscripts_scm'
# UNION ALL
# SELECT 'Mapping' AS layer, COUNT(*) AS record_count FROM _exponent.omop_mapping.source_to_observation_period WHERE source_system = 'allscripts_scm'
# UNION ALL
# SELECT 'Gold' AS layer, COUNT(*) AS record_count FROM _exponent.omop.observation_period

In [0]:
# %sql
# -- Verify gold
# SELECT * FROM _exponent.omop.observation_period
# LIMIT 10

In [0]:
%sql
SELECT DISTINCT
observation_period.person_id 
FROM _exponent.omop_scm.observation_period
ANTI JOIN _exponent.omop_scm.person
ON observation_period.person_id = person.person_id